# Controlled 4-state 2BSDE: two-dimensional nonlinear drift, observation- and posterior-dependent benchmark

The dynamics, control set and time horizon are:
$$
X\in\{(-1,-1),(-1,1),(1,-1),(1,1)\},\qquad
dY_t=h(Y_t,u_t,X)\,dt+dW_t,
\qquad u_t\in[-1,1]^2,\qquad T=1.
$$
The observation $Y$, hidden state $X$, control $u$ and Brownian motion $W$ are
two-dimensional. The hidden state is fixed but unknown. Its posterior is
$p_t^i=\mathbb P(X=x_i\mid\mathcal Y_t)$, with the four support points ordered
as above. Define
$$
L=\frac13\begin{pmatrix}2&1\\1&2\end{pmatrix},\qquad
\lambda_1(y)=\frac34+\frac14\tanh(y_1+y_2),\qquad
\lambda_2(y)=\frac34+\frac14\tanh(y_1-y_2).
$$
The drift is
$$
h_j(y,u,x)=\lambda_j(y)\bigl((Lx)_j-(Lu)_j\bigr)^2,
\qquad j=1,2.
$$
Each observation channel depends nonlinearly on both observation coordinates,
both hidden coordinates and both controls. The control also changes what can
be learned about $X$. The same two-dimensional innovation Brownian motion $I$
drives the coupled state:
$$
dY_t=\bar h(Y_t,p_t,u_t)\,dt+dI_t,\qquad
dp_t^i=p_t^i\bigl(h(Y_t,u_t,x_i)-\bar h(Y_t,p_t,u_t)\bigr)^\top dI_t,
$$
$$
\bar h(y,p,u)=\sum_i p_i h(y,u,x_i).
$$
The state used by the 2BSDE is $S=(Y_1,Y_2,p_1,p_2,p_3,p_4)$; no additional
sensor noise is introduced. Four posterior probabilities have three independent
coordinates, even though the hidden variable itself is two-dimensional.

Define $\xi=LX$ and the posterior moments
$$
m=\mathbb E_p[X],\qquad \Sigma=\operatorname{Cov}_p(X),\qquad
\mu_j=\mathbb E_p[\xi_j],\qquad s_j=\mathbb E_p[\xi_j^2],
$$
$$
a_j=\operatorname{Cov}_p(X,\xi_j),\qquad
c_j=\operatorname{Cov}_p(X,\xi_j^2),\qquad j=1,2.
$$
Here $\mu_j,s_j$ are scalars and $a_j,c_j\in\mathbb R^2$. Set
$$
q_j=\tanh y_j,\qquad
d_j=\lambda_jq_j-2\lambda_j^2\|a_j\|^2,\qquad
b_j=\lambda_jq_j\mu_j-\lambda_j^2 a_j^\top c_j,
$$
$$
Q(y,p)=8I_2+L^\top\operatorname{diag}(d_1,d_2)L,\qquad
B(y,p)=L^\top(b_1,b_2)^\top.
$$
The running and terminal costs are
$$
k(y,p,u)=8\|u\|^2
+\frac12\sum_{j=1}^2\left[(q_j-\lambda_js_j)^2+1-\lambda_j^2s_j^2
+\lambda_j^2\|c_j\|^2\right]+B^\top Q^{-1}B,
$$
$$
g(y,p)=\sum_{j=1}^2\log\cosh y_j+\tfrac12\operatorname{tr}\Sigma.
$$
We minimise
$$
J_{t,y,p}(u)=\mathbb E_{t,y,p}\left[
\int_t^T k(Y_r,p_r,u_r)\,dr+g(Y_T,p_T)\right].
$$
The terminal cost penalises displacement from $y=(0,0)$ and remaining
uncertainty. The running cost contains quadratic control effort and a
state-dependent term chosen to make the solution explicit. Since
$1/2<\lambda_j<1$, $0\le s_j\le1$ and $Q\succeq3I_2$, both costs are
nonnegative. This is a constructed benchmark, rather than a claim that an
arbitrary tracking cost has the same solution.

The exact value and optimal feedback are
$$
\boxed{V(t,y,p)=\sum_{j=1}^2\log\cosh y_j+
\tfrac12\operatorname{tr}\Sigma+2(T-t),\qquad
u^*(t,y,p)=Q(y,p)^{-1}B(y,p).}
$$
The feedback has no explicit time dependence, but changes along a path through
$Y_t$ and $p_t$. The functions below retain the same time arguments as the
original notebook. Since $\|B\|\le2\sqrt2$ and $Q\succeq3I_2$,
$\|u^*\|\le2\sqrt2/3<1$; the optimum needs no clipping. The code uses a
linear solve, rather than explicitly forming $Q^{-1}$.

For $y_0=(0,0)$, $p_0=(0.1,0.2,0.3,0.4)$,
$$
V_0=2.9,\qquad
u_0^*=\frac1{6761905}(-53522,-59194)^\top
\simeq(-0.00791523,-0.00875404)^\top.
$$
We use an asymmetric prior. The value and gradient depend on the state. The
observation-observation Hessian also depends on the state; the
posterior-posterior block is constant and the mixed block is zero.

In [1]:
from pathlib import Path
import math
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from two_bsde.config import ModelConfig, HamiltonianConfig, TrainingConfig
from two_bsde.problem import BayesianControlProblem
from two_bsde.geometry import make_simplex_tangent_basis, SimplexGeometry
from two_bsde.hamiltonian import HamiltonianOptimizer
from two_bsde.twoBsdeStructure import GeometricTimeBlockAdaptive2BSDE
from two_bsde.dynamics import split_state, state_dynamics
from two_bsde.training import train_time_blocks
from two_bsde.evaluation import evaluate_time_blocks

DTYPE = torch.float64
DEVICE = torch.device("cpu")

torch.manual_seed(42)
np.random.seed(42)

In [2]:
config = ModelConfig(
    horizon=1.0,
    hidden_dim=2,
    observation_dim=2,
    control_dim=2,
    num_hidden_states=4,
    num_time_steps=30,
    network_width=64,
    network_depth=5,
    value_initial_guess=0.01,
    gamma_initial_scale=0.01,
)

ham_config = HamiltonianConfig(
    num_starts=12,
    num_steps=30,
    learning_rate=5e-2,
)

training_config = TrainingConfig(
    epochs=1500,
    batch_size=128,
    learning_rate=1e-4,
    gradient_clip=10.0,
    value_loss_weight=1.0,
    gradient_loss_weight=1.0,
)

basis = make_simplex_tangent_basis(
    config.num_hidden_states,
    dtype=DTYPE,
    device=DEVICE,
)

geometry = SimplexGeometry(
    observation_dim=config.observation_dim,
    basis=basis,
)

print("state_dim =", config.state_dim)
print("tangent_dim =", config.tangent_dim)

state_dim = 6
tangent_dim = 3


## Concrete problem

In [3]:
class ControlledFourStateProblem(BayesianControlProblem):
    def __init__(self, config):
        super().__init__(config)
        if config.horizon != 1.0:
            raise ValueError("This benchmark uses horizon T=1.")
        if getattr(config, "observation_noise_std", 1.0) != 1.0:
            raise ValueError("This analytic benchmark uses observation noise sigma=I_2.")

        self.x_support = torch.tensor(
            [[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]],
            dtype=DTYPE, device=DEVICE,
        )
        self.L = torch.tensor([[2.0, 1.0], [1.0, 2.0]], dtype=DTYPE, device=DEVICE) / 3.0
        self.initial_observation = torch.tensor([0.0, 0.0], dtype=DTYPE, device=DEVICE)
        self.initial_posterior = torch.tensor([0.10, 0.20, 0.30, 0.40], dtype=DTYPE, device=DEVICE)
        self.control_low = torch.tensor([-1.0, -1.0], dtype=DTYPE, device=DEVICE)
        self.control_high = torch.tensor([1.0, 1.0], dtype=DTYPE, device=DEVICE)

        # Retained for compatibility with the original problem setup.
        # The time-block model instead applies each path's own minimiser.
        self.auxiliary_control_path = torch.zeros(
            config.num_time_steps, config.control_dim, dtype=DTYPE, device=DEVICE,
        )
        self.validate(config)

    @staticmethod
    def signal_level(observation):
        directions = torch.stack(
            (observation[..., 0] + observation[..., 1],
             observation[..., 0] - observation[..., 1]), dim=-1,
        )
        return 0.75 + 0.25 * torch.tanh(directions)

    def sensor(self, control, observation):
        L = self.L.to(control)
        xi = self.x_support.to(control) @ L.T                 # [4, 2]
        w = control @ L.T                                    # [..., 2]
        lam = self.signal_level(observation)                   # [..., 2]
        # [..., 4, 2], including separate path and candidate-control axes.
        return (xi - w.unsqueeze(-2)).square() * lam.unsqueeze(-2)

    @staticmethod
    def log_cosh(y):
        # Stable at large |y| and twice differentiable at y=0.
        return torch.logaddexp(y, -y) - math.log(2.0)

    def posterior_terms(self, state):
        p = state[..., self.config.observation_dim:]
        x = self.x_support.to(state)
        xi = x @ self.L.to(state).T
        m = p @ x
        trace_cov = (p * x.square().sum(dim=-1)).sum(dim=-1) - m.square().sum(dim=-1)
        mu, s = p @ xi, p @ xi.square()
        # a and c: [..., hidden coordinate, observation coordinate].
        a = torch.einsum("...i,id,ij->...dj", p, x, xi) - m.unsqueeze(-1) * mu.unsqueeze(-2)
        c = torch.einsum("...i,id,ij->...dj", p, x, xi.square()) - m.unsqueeze(-1) * s.unsqueeze(-2)
        return m, trace_cov, mu, s, a, c

    def hamiltonian_terms(self, state):
        _, _, mu, s, a, c = self.posterior_terms(state)
        y = state[..., :self.config.observation_dim]
        q, lam = torch.tanh(y), self.signal_level(y)
        # Each column is a_j or c_j, so sum over the hidden-coordinate axis.
        d = lam * q - 2.0 * lam.square() * a.square().sum(dim=-2)
        b = lam * q * mu - lam.square() * (a * c).sum(dim=-2)
        L = self.L.to(state)
        Q = 8.0 * torch.eye(2, dtype=state.dtype, device=state.device)
        Q = Q + torch.einsum("ji,...j,jk->...ik", L, d, L)
        B = b @ L
        state_cost = 0.5 * (
            (q - lam * s).square() + 1.0 - (lam * s).square()
            + lam.square() * c.square().sum(dim=-2)
        ).sum(dim=-1)
        return Q, B, state_cost

    def running_cost(self, state, control, *, time=None):
        # This example is autonomous; retain time for the package interface.
        Q, B, state_cost = self.hamiltonian_terms(state)
        Q_inv_B = torch.linalg.solve(Q, B.unsqueeze(-1)).squeeze(-1)
        return 8.0 * control.square().sum(dim=-1) + state_cost + (B * Q_inv_B).sum(dim=-1)

    def terminal_cost(self, state):
        y, p = state[..., :self.config.observation_dim], state[..., self.config.observation_dim:]
        x = self.x_support.to(state)
        m = p @ x
        trace_cov = (p * x.square().sum(dim=-1)).sum(dim=-1) - m.square().sum(dim=-1)
        return self.log_cosh(y).sum(dim=-1) + 0.5 * trace_cov


problem = ControlledFourStateProblem(config)
S0 = torch.cat((problem.initial_observation, problem.initial_posterior))
print("S0 =", S0)
print("hidden support =", problem.x_support)
print("Control: each path uses its own Hamiltonian minimiser at every fine step.")

S0 = tensor([0.0000, 0.0000, 0.1000, 0.2000, 0.3000, 0.4000], dtype=torch.float64)
hidden support = tensor([[-1., -1.],
        [-1.,  1.],
        [ 1., -1.],
        [ 1.,  1.]], dtype=torch.float64)
Control: each path uses its own Hamiltonian minimiser at every fine step.


## Analytic solution

The derivatives below use the posterior tangent projection
$P=I-\mathbf1\mathbf1^\top/4$. Let $\mathsf X\in\mathbb R^{4\times2}$ have
rows $x_i^\top$, and let $r_i=\|x_i\|^2$. Since
$\operatorname{tr}\Sigma=\sum_i p_i r_i-\|m\|^2$,
$$
Z_y=(\tanh y_1,\tanh y_2)^\top,\qquad
Z_p=P\left(\tfrac12r-\mathsf X m\right)=-\mathsf X m,
$$
$$
\Gamma_{yy}=\operatorname{diag}(1-\tanh^2y_1,1-\tanh^2y_2),\qquad
\Gamma_{yp}=0,\qquad
\Gamma_{pp}=-P\mathsf X\mathsf X^\top P=-\mathsf X\mathsf X^\top.
$$
The last equalities use $r=2\mathbf1$ and $P\mathsf X=\mathsf X$ for the
chosen support. Here $Z$ denotes the state gradient in the package's convention.
The posterior Hessian is $-\mathsf X\mathsf X^\top$, not $-P$.

The mixed block is zero because $V$ is additive in $y$ and $p$, even though
$Y$ and $p$ share the same Brownian motion. The full Hessian contraction is
still used below. At $T$, the same gradient and Hessian are those of $g$.
The nonlinear observation dependence in $h$ changes the dynamics and feedback;
the chosen running cost preserves this explicit value function.

In [4]:
def exact_value(time, state):
    time = torch.as_tensor(time, dtype=state.dtype, device=state.device)
    return problem.terminal_cost(state) + 2.0 * (config.horizon - time)


def exact_gradient(time, state):
    x = problem.x_support.to(state)
    p = state[..., config.observation_dim:]
    m = p @ x
    ambient = torch.zeros_like(state)
    ambient[..., :config.observation_dim] = torch.tanh(state[..., :config.observation_dim])
    ambient[..., config.observation_dim:] = 0.5 * x.square().sum(dim=-1) - m @ x.T
    return geometry.project_gradient(ambient)


def exact_hessian(time, state):
    x = problem.x_support.to(state)
    shape = state.shape[:-1] + (config.state_dim, config.state_dim)
    gamma = torch.zeros(shape, dtype=state.dtype, device=state.device)
    gamma[..., :config.observation_dim, :config.observation_dim] = torch.diag_embed(
        1.0 - torch.tanh(state[..., :config.observation_dim]).square()
    )
    projected_x = geometry.projector @ x
    gamma[..., config.observation_dim:, config.observation_dim:] = -projected_x @ projected_x.T
    return gamma


def exact_control(time, state):
    Q, B, _ = problem.hamiltonian_terms(state)
    return torch.linalg.solve(Q, B.unsqueeze(-1)).squeeze(-1)


V0_exact = exact_value(0.0, S0)
u0_exact = exact_control(0.0, S0)
print("exact V0 =", V0_exact.item())
print("exact u0* =", u0_exact.cpu().numpy())
assert abs(V0_exact.item() - 2.9) < 1e-12
u0_reference = torch.tensor([-53522.0, -59194.0], dtype=DTYPE, device=DEVICE) / 6761905.0
assert torch.allclose(u0_exact, u0_reference, atol=1e-12, rtol=0.0)

exact V0 = 2.9
exact u0* = [-0.00791523 -0.00875404]


## Verify the analytic HJB

Put $w=Lu$. The posterior mean satisfies
$$
dm_t=\begin{pmatrix}
\lambda_1(c_1-2w_1a_1)&\lambda_2(c_2-2w_2a_2)
\end{pmatrix}dI_t.
$$
Since $\mathbb E_{p_t}[\|X\|^2]$ is a martingale, Itô's formula gives
$$
\mathcal L^u\left(\tfrac12\operatorname{tr}\Sigma\right)
=-\tfrac12\sum_{j=1}^2\lambda_j^2\|c_j-2w_ja_j\|^2.
$$
Consequently,
$$
\mathcal L^uV=\sum_{j=1}^2\left[
q_j\lambda_j(s_j-2\mu_jw_j+w_j^2)
+\tfrac12(1-q_j^2)-\tfrac12\lambda_j^2\|c_j-2w_ja_j\|^2
\right].
$$
The generator differentiates $V$, not the drift coefficient, so no derivatives
of $\lambda_j$ enter this expression. Using $\partial_tV=-2$, the terms
independent of $u$ cancel against the chosen running cost, leaving
$$
\partial_tV+\mathcal L^uV+k
=u^\top Qu-2B^\top u+B^\top Q^{-1}B
=(u-u^*)^\top Q(u-u^*).
$$
As $Q\succeq3I_2$ and $u^*\in[-1,1]^2$, the HJB minimum is zero and is
attained at $u^*=Q^{-1}B$. Also $V(T,y,p)=g(y,p)$. Applying Itô's formula
and taking expectations yields the verification identity
$$
J_{t,y,p}(u)-V(t,y,p)
=\mathbb E_{t,y,p}\int_t^T
(u_r-u^*(Y_r,p_r))^\top Q(Y_r,p_r)(u_r-u^*(Y_r,p_r))\,dr\ge0.
$$
The coefficients and derivatives in this identity are bounded, so the stochastic
integral has zero expectation. Equality holds for the displayed feedback.

The checks below use the package dynamics and the full Hessian. They verify
the identity for several states, times and controls, the terminal condition,
and the analytic derivatives against autograd.

In [5]:
def analytic_hjb_expression(time, state, control):
    drift, sigma, _ = state_dynamics(
        state, control, problem=problem, config=config, geometry=geometry,
    )
    time_derivative = -2.0 * torch.ones_like(state[..., 0])
    running = problem.running_cost(state, control, time=time)
    z = exact_gradient(time, state)
    gamma = exact_hessian(time, state)
    first_order = (drift * z).sum(dim=-1)
    # Full contraction; the analytic mixed block happens to be zero here.
    second_order = 0.5 * torch.einsum("...ia,...ij,...ja->...", sigma, gamma, sigma)
    return time_derivative + running + first_order + second_order


test_states = torch.tensor(
    [[0.0, 0.0, 0.10, 0.20, 0.30, 0.40],
     [0.4, -0.2, 0.55, 0.15, 0.20, 0.10],
     [-0.6, 0.7, 0.05, 0.10, 0.25, 0.60]],
    dtype=DTYPE, device=DEVICE,
)

for time in [0.0, 0.25, 0.5, 0.9, 1.0]:
    u_exact = exact_control(time, test_states)
    hjb_at_exact = analytic_hjb_expression(time, test_states, u_exact)
    print(f"t={time:.2f}: analytic control = {u_exact.cpu().numpy()}")
    print("         HJB at analytic control =", hjb_at_exact.cpu().numpy())
    assert torch.allclose(hjb_at_exact, torch.zeros_like(hjb_at_exact), atol=1e-12)

    Q, _, _ = problem.hamiltonian_terms(test_states)
    assert torch.all(torch.linalg.eigvalsh(Q) >= 3.0 - 1e-12)
    assert torch.all(torch.linalg.vector_norm(u_exact, dim=-1) <= 2.0 * math.sqrt(2.0) / 3.0)
    for u_value in [[-1.0, -1.0], [-1.0, 1.0], [0.0, 0.0], [0.5, -0.5], [1.0, 1.0]]:
        u = torch.tensor(u_value, dtype=DTYPE, device=DEVICE).expand(len(test_states), -1)
        residual = analytic_hjb_expression(time, test_states, u)
        error = u - u_exact
        square = torch.einsum("...i,...ij,...j->...", error, Q, error)
        assert torch.allclose(residual, square, atol=1e-12)
        assert torch.all(residual >= -1e-12)
        assert torch.all(problem.running_cost(test_states, u, time=time) >= -1e-12)

assert torch.allclose(exact_value(1.0, test_states), problem.terminal_cost(test_states))
assert torch.allclose(
    exact_gradient(1.0, test_states),
    geometry.project_gradient(problem.terminal_gradient(test_states)),
)
for state in test_states:
    gradient_ad = torch.autograd.functional.jacobian(problem.terminal_cost, state)
    hessian_ad = torch.autograd.functional.hessian(problem.terminal_cost, state)
    assert torch.allclose(exact_gradient(1.0, state), geometry.project_gradient(gradient_ad), atol=1e-12)
    assert torch.allclose(exact_hessian(1.0, state), geometry.project_hessian(hessian_ad), atol=1e-12)
print("analytic HJB, terminal and derivative checks: PASSED")

t=0.00: analytic control = [[-0.00791523 -0.00875404]
 [ 0.00398013  0.01335395]
 [-0.02208437 -0.00375726]]
         HJB at analytic control = [0.00000000e+00 2.22044605e-16 1.11022302e-16]
t=0.25: analytic control = [[-0.00791523 -0.00875404]
 [ 0.00398013  0.01335395]
 [-0.02208437 -0.00375726]]
         HJB at analytic control = [0.00000000e+00 2.22044605e-16 1.11022302e-16]
t=0.50: analytic control = [[-0.00791523 -0.00875404]
 [ 0.00398013  0.01335395]
 [-0.02208437 -0.00375726]]
         HJB at analytic control = [0.00000000e+00 2.22044605e-16 1.11022302e-16]
t=0.90: analytic control = [[-0.00791523 -0.00875404]
 [ 0.00398013  0.01335395]
 [-0.02208437 -0.00375726]]
         HJB at analytic control = [0.00000000e+00 2.22044605e-16 1.11022302e-16]
t=1.00: analytic control = [[-0.00791523 -0.00875404]
 [ 0.00398013  0.01335395]
 [-0.02208437 -0.00375726]]
         HJB at analytic control = [0.00000000e+00 2.22044605e-16 1.11022302e-16]
analytic HJB, terminal and derivative checks:

## Build the time-block 2BSDE model

Use the same 3 shared blocks and 30 fine time steps. All paths start from the
initial state above. Each path applies its own fine-step Hamiltonian minimiser.

In [6]:
hamiltonian_optimizer = HamiltonianOptimizer(
    problem=problem,
    model_config=config,
    optimizer_config=ham_config,
    geometry=geometry,
)

model = GeometricTimeBlockAdaptive2BSDE(
    problem=problem,
    config=config,
    geometry=geometry,
    hamiltonian_optimizer=hamiltonian_optimizer,
    num_blocks=3,
).to(device=DEVICE, dtype=DTYPE)

print("number of time blocks =", model.num_blocks)


number of time blocks = 3


## Train the time-block 2BSDE


In [ ]:
history = train_time_blocks(
    model,
    training_config=training_config,
    sample_start_times=False,  # start at t=0 
    # boundary_weight=0.0,     # uncomment to disable the boundary penalty
    verbose=True,
)

epoch=   1  start_step=  0  total_loss=9.790e+00  terminal_value=9.047e+00  terminal_gradient=3.103e-01  consistency=4.319e-01  boundary=6.455e-05  mean_start_value=0.014777


In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(history.total_loss, label="total")
plt.semilogy(history.terminal_value_loss, label="terminal value")
plt.semilogy(history.terminal_gradient_loss, label="terminal gradient")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Time-block 2BSDE training")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## Compare the learned initial value with the analytic solution


In [ ]:
with torch.no_grad():
    V0_learned = model.value(0.0, S0)

print("VALUE")
print(" learned:", V0_learned.item())
print(" exact:  ", V0_exact.item())
print(" abs err:", abs(V0_learned.item() - V0_exact.item()))


## Test the inner Hamiltonian optimizer with the analytic derivatives

Test several posteriors and times. For two-dimensional controls the optimizer
uses projected Adam from several starting points, so compare with the analytic
optimum using control errors and the Hamiltonian gap. There is no scalar control
grid or half-grid-spacing tolerance in this case. The finite search need not
reach the exact optimum. This separates the control-search error from errors
in the learned $Z$ and $\Gamma$.

With the exact derivatives, the Hamiltonian gap is
$(u^{\mathrm{num}}-u^*)^\top Q(u^{\mathrm{num}}-u^*)$.

In [ ]:
state_batch = torch.tensor(
    [[0.0, 0.0, 0.10, 0.20, 0.30, 0.40],
     [0.2, -0.4, 0.10, 0.15, 0.25, 0.50],
     [-0.3, 0.5, 0.55, 0.20, 0.15, 0.10]],
    dtype=DTYPE, device=DEVICE,
)

print("control search: projected Adam")
print("starts =", ham_config.num_starts, "; steps =", ham_config.num_steps)

with torch.no_grad():
    for time in [0.0, 0.25, 0.5, 0.9]:
        h_numeric, u_from_exact_derivatives = hamiltonian_optimizer.minimise(
            state_batch,
            exact_gradient(time, state_batch),
            exact_hessian(time, state_batch),
            time=time,
        )
        u_analytic = exact_control(time, state_batch)
        h_analytic = hamiltonian_optimizer.evaluate(
            state_batch, exact_gradient(time, state_batch),
            exact_hessian(time, state_batch), u_analytic, time=time,
        )
        error = u_from_exact_derivatives - u_analytic
        Q, _, _ = problem.hamiltonian_terms(state_batch)
        gap = h_numeric - h_analytic
        quadratic_gap = torch.einsum("...i,...ij,...j->...", error, Q, error)
        print(f"t={time:.2f}")
        print(" numerical u* =", u_from_exact_derivatives.cpu().numpy())
        print(" analytic  u* =", u_analytic.cpu().numpy())
        print(" max abs error by component =", error.abs().amax(dim=0).cpu().numpy())
        print(" Hamiltonian gap =", gap.cpu().numpy())
        assert torch.all(u_from_exact_derivatives >= problem.control_low - 1e-12)
        assert torch.all(u_from_exact_derivatives <= problem.control_high + 1e-12)
        assert torch.allclose(h_analytic, torch.full_like(h_analytic, 2.0), atol=1e-12)
        assert torch.allclose(gap, quadratic_gap, atol=1e-12)
        assert torch.all(gap >= -1e-12)

## Fresh-path 2BSDE evaluation: paired analytic comparisons

For each simulated numerical state $S_k^{(b)}=(Y_k^{(b)},p_k^{(b)})$, compare
$$
u_k^{\mathrm{num},(b)}\quad\hbox{with}\quad
u^*(t_k,Y_k^{(b)},p_k^{(b)}).
$$
The analytic feedback is evaluated at **the same observation, posterior and time**,
so no independent analytic trajectory is needed. Controls and Hessians use the
state at the **start** of each step; value and gradient paths also include $T$.

We report paired MAE and RMSE for each control component, alongside individual
paired paths and ensemble summaries. The 3D plots use $(t,u_1,u_2)$ and hold
each control fixed over its fine time step. The analytic 10–90% bands measure
variation across paths, not confidence intervals or acceptable error ranges.
They are plotted separately for $u_1$ and $u_2$; the numerical mean need not
lie inside an analytic band. Both edges are plotted explicitly and the vertical
axes are allowed to autoscale. The exact controls can be negative or cross zero,
so we use absolute errors rather than relative errors.

The analytic reference is exact for the continuous-time problem; the numerical
state paths use the existing Euler scheme and posterior stabilisation.

In [ ]:
result = evaluate_time_blocks(model, num_paths=500, evaluate_policy=False)
paths = result.paths

with torch.no_grad():
    times = paths["time_path"]                    # [steps+1]
    control_times = times[:-1]                    # [steps]
    states = paths["S_path"]                      # [paths, steps+1, state_dim]
    u_star_path = paths["u_star_path"]            # [paths, steps, 2]
    u_exact_path = exact_control(control_times, states[:, :-1])
    control_error = u_star_path - u_exact_path
    mean_u_by_time = u_star_path.mean(dim=0)       # [steps, 2]
    mean_exact_u_by_time = u_exact_path.mean(dim=0)
    mae_u_by_time = control_error.abs().mean(dim=0)
    rmse_u_by_time = control_error.square().mean(dim=0).sqrt()
    q10_exact, q90_exact = torch.quantile(
        u_exact_path, torch.tensor([0.1, 0.9], dtype=DTYPE, device=DEVICE), dim=0,
    )

    V_exact_path = exact_value(times, states)
    value_head_error = paths["value_path"] - V_exact_path
    propagated_value_error = paths["R_path"] - V_exact_path

assert torch.equal(paths["u_aux_path"], paths["u_star_path"])
print("PAIRED CONTROL COMPARISON AT THE SAME STATES")
print(" mean absolute error by component =", control_error.abs().mean(dim=(0, 1)).cpu().numpy())
print(" root mean square error by component =", control_error.square().mean(dim=(0, 1)).sqrt().cpu().numpy())
print(" maximum absolute error by component =", control_error.abs().amax(dim=(0, 1)).cpu().numpy())
print(" maximum analytic 10-90% band width by component =", (q90_exact - q10_exact).amax(dim=0).cpu().numpy())
print("PAIRED VALUE COMPARISON AT THE SAME STATES")
print(" value-network RMSE =", value_head_error.square().mean().sqrt().item())
print(" propagated R RMSE  =", propagated_value_error.square().mean().sqrt().item())
print(" posterior correction rate =", result.diagnostics["posterior_correction_rate"])

In [ ]:
path_id = 0

with torch.no_grad():
    Z_learned = paths["Z_path"][path_id].detach().cpu()
    Z_exact = exact_gradient(times, states[path_id]).detach().cpu()

time_numpy = times.detach().cpu().numpy()
labels = ["y1", "y2", "p1", "p2", "p3", "p4"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(time_numpy, Z_learned[:, i].numpy(), label="Numerical Z")
    ax.plot(time_numpy, Z_exact[:, i].numpy(), "--", color="black", label="Analytic at same state")
    ax.set_title(f"Z[{labels[i]}]")
    ax.set_xlabel("t")
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle(f"Gradient comparison along simulated path {path_id}")
plt.tight_layout()
plt.show()

In [ ]:
with torch.no_grad():
    Gamma0_learned = model.blocks[0].gamma(0.0, S0)
    Gamma0_exact = exact_hessian(0.0, S0)

print("HESSIAN AT t=0")
print(" learned:\n", Gamma0_learned)
print(" exact:\n", Gamma0_exact)
print(" Frobenius error:", torch.linalg.matrix_norm(Gamma0_learned - Gamma0_exact, ord="fro").item())

In [ ]:
with torch.no_grad():
    # Gamma at step k belongs to the state at the START of that step.
    Gamma_learned = paths["Gamma_path"]           # [paths, steps, 6, 6]
    Gamma_exact = exact_hessian(control_times, states[:, :-1])
    error = torch.linalg.matrix_norm(
        Gamma_learned - Gamma_exact, ord="fro", dim=(-2, -1),
    )                                           # [paths, steps]

print("Mean Hessian error:", error.mean().item())
plt.figure(figsize=(7, 4))
plt.plot(control_times.cpu().numpy(), error.mean(dim=0).cpu().numpy(), marker="o")
plt.xlabel("t")
plt.ylabel("Mean Frobenius error")
plt.title("Learned Hessian vs analytic Hessian")
plt.grid(alpha=0.25)
plt.show()

In [ ]:
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

time_grid = control_times.detach().cpu().numpy()

# Individual paired paths show differences hidden by ensemble means.
# Repeated endpoints display controls held fixed over [t_k, t_{k+1}).
step_times = np.repeat(times.detach().cpu().numpy(), 2)[1:-1]
fig = plt.figure(figsize=(15, 5))
for path_id in range(3):
    ax = fig.add_subplot(1, 3, path_id + 1, projection="3d")
    numerical = np.repeat(u_star_path[path_id].detach().cpu().numpy(), 2, axis=0)
    analytic = np.repeat(u_exact_path[path_id].detach().cpu().numpy(), 2, axis=0)
    ax.plot(step_times, numerical[:, 0], numerical[:, 1],
            color="tab:orange", label="Numerical")
    ax.plot(step_times, analytic[:, 0], analytic[:, 1], linestyle="--",
            color="black", label="Analytic at same state")
    ax.set_title(f"Path {path_id}")
    ax.set_xlabel("t", labelpad=6)
    ax.set_ylabel("u1", labelpad=8)
    ax.set_zlabel("u2", labelpad=10)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_major_locator(MaxNLocator(4))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
    ax.zaxis.set_major_formatter(FormatStrFormatter("%.3f"))
    ax.tick_params(labelsize=8, pad=1)
    ax.set_xlim(float(times[0]), float(times[-1]))
    ax.view_init(elev=24, azim=-55)
    ax.set_box_aspect((1.3, 1.0, 1.0))
    ax.legend(fontsize=8)
fig.suptitle("Paired feedback controls along numerical paths")
fig.subplots_adjust(left=0.02, right=0.93, bottom=0.10, top=0.87, wspace=0.30)
plt.show()

Ensemble, 500 simulated paths, for each control component $j=1,2$,
$$
    e_{i,j}(t)=u_{i,j}^{\mathrm{num}}(t)-u_j^*(t,S_i(t)),
    \qquad
    \operatorname{MAE}_j(t)=\frac{1}{N}\sum_{i=1}^N\left|e_{i,j}(t)\right|.
$$
$$
    \operatorname{RMSE}_j(t)=\sqrt{\frac{1}{N}\sum_{i=1}^N e_{i,j}(t)^2}.
$$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for j, ax in enumerate(axes):
    ax.plot(time_grid, mean_u_by_time[:, j].cpu().numpy(), marker="o", color="tab:orange",
            label="Mean numerical control")
    ax.plot(time_grid, mean_exact_u_by_time[:, j].cpu().numpy(), "--", color="black",
            label="Mean analytic control at same states")
    ax.fill_between(time_grid, q10_exact[:, j].cpu().numpy(), q90_exact[:, j].cpu().numpy(),
                    color="tab:blue", alpha=0.25, label="Analytic 10–90% path band")
    ax.plot(time_grid, q10_exact[:, j].cpu().numpy(), color="tab:blue", linewidth=0.8)
    ax.plot(time_grid, q90_exact[:, j].cpu().numpy(), color="tab:blue", linewidth=0.8)
    ax.set_xlabel("t")
    ax.set_ylabel(f"u{j + 1}")
    ax.set_title(f"Control ensemble comparison: u{j + 1}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for j, ax in enumerate(axes):
    ax.plot(time_grid, mae_u_by_time[:, j].cpu().numpy(), marker="o", label="Paired MAE")
    ax.plot(time_grid, rmse_u_by_time[:, j].cpu().numpy(), marker="s", label="Paired RMSE")
    ax.set_xlabel("t")
    ax.set_ylabel("Absolute control error")
    ax.set_title(f"Control errors at matching times and states: u{j + 1}")
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()